[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fvalenzuelag/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/Mistral/RA2/IL2.2/2-memory-agent-advanced.ipynb)


# 2. Sistemas de Memoria Avanzados en LangChain

## Objetivos de Aprendizaje
- Comprender las limitaciones de la gestión manual de la memoria.
- Aprender a usar las clases de memoria de LangChain para automatizar el historial de chat.
- Implementar `ConversationBufferMemory` para un historial completo.
- Implementar `ConversationBufferWindowMemory` para limitar el tamaño del historial.
- Implementar `ConversationSummaryMemory` para resumir conversaciones y ahorrar tokens.
- Entender el puente hacia la planificación de agentes.

## El Problema con la Memoria Manual

En el notebook anterior, le dimos memoria a nuestro agente gestionando manualmente una lista `chat_history`. Aunque funcional, este enfoque tiene dos grandes inconvenientes:

1.  **Gestión Tediosa**: Actualizar la lista manualmente después de cada interacción es propenso a errores y no es escalable.
2.  **Límite de Contexto**: Enviar el historial completo en cada llamada consume tokens rápidamente. En conversaciones largas, esto puede exceder el límite de contexto del modelo, provocando errores o un alto costo.

LangChain resuelve esto con un sistema de **clases de Memoria** que automatizan el proceso y ofrecen estrategias para gestionar historiales largos.

### 1. Instalación y Configuración

In [1]:
# --- Instalación de dependencias (se ejecuta solo en Google Colab) ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys
if "google.colab" in sys.modules:
    !pip install -q langchain langchain-classic langchain-openai python-dotenv wikipedia


In [2]:
# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os
try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    # Solo LLM_API_KEY es obligatorio. Los demás son opcionales: defínelos como
    # Secrets únicamente si quieres usar otro proveedor o modelo.
    for _k in ("LLM_API_KEY", "GOOGLE_API_KEY", "LANGSMITH_API_KEY",
               "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass                                # el Secret no existe: se usa el default
    os.environ.setdefault("LLM_BASE_URL", "https://api.mistral.ai/v1")
    os.environ.setdefault("LLM_MODEL", "mistral-small-latest")
    os.environ.setdefault("LLM_MODEL_SMALL", "ministral-8b-latest")
except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model=os.getenv("LLM_MODEL", "mistral-small-latest"),
        openai_api_base=os.environ.get("LLM_BASE_URL"),
        openai_api_key=os.environ.get("LLM_API_KEY"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

✅ LLM de LangChain configurado.


### 2. Agente y Herramientas (Sin Cambios)

Reutilizamos el mismo agente y herramientas. La innovación estará en cómo gestionamos la memoria.

In [3]:
# LangChain v1: langchain.agents.create_agent (o langgraph)
from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Este prompt antes se descargaba con `hub.pull("hwchase17/openai-tools-agent")`.
# LangChain ahora bloquea por defecto la descarga de prompts públicos, porque
# pueden contener objetos serializados no confiables. Definirlo aquí evita esa
# dependencia de red y, sobre todo, deja a la vista qué le estamos diciendo al
# agente: instrucción de sistema, historial, entrada del usuario y el
# "scratchpad" donde el agente va anotando sus llamadas a herramientas.
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")

✅ Agente y herramientas listos.


### 3. Automatizando con `ConversationBufferMemory`

Esta es la memoria más básica. Simplemente almacena todos los mensajes en una variable (el "buffer") y los pasa en cada llamada. Replica lo que hicimos manualmente, pero de forma automática.

In [4]:
# LangChain v1: persistencia de LangGraph (RunnableWithMessageHistory, el paso
# intermedio que usa este notebook, también quedó deprecado)
from langchain_classic.memory import ConversationBufferMemory

# memory_key debe coincidir con la variable de entrada en el prompt ('chat_history')
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

def chat_with_agent_buffer(query: str):
    # Carga el historial de la memoria
    history = memory.load_memory_variables({})["chat_history"]
    
    # Invoca al agente con el historial
    response = agent_executor.invoke({"input": query, "chat_history": history})
    
    # Guarda el nuevo par de pregunta/respuesta en la memoria
    memory.save_context({"input": query}, {"output": response["output"]})
    
    return response["output"]

/var/folders/hc/s32qf6c501b8c_vjpqkvxdpw0000gn/T/ipykernel_8889/170025402.py:6: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


In [5]:
print("Primera pregunta...")
response1 = chat_with_agent_buffer("Háblame del planeta Marte")
print(f"Respuesta 1: {response1}")

print("Segunda pregunta de seguimiento...")
response2 = chat_with_agent_buffer("¿Tiene alguna luna?")
print(f"Respuesta 2: {response2}")

Primera pregunta...


> Entering new AgentExecutor chain...



Invoking: `get_wikipedia_summary` with `{'query': 'Marte planeta'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)


Invoking: `get_wikipedia_summary` with `{'query': 'Marte'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)

...

Ocurrió un error: Expecting value: line 1 column 1 (char 0)

Parece que hay un problema técnico con la herramienta para obtener información de Wikipedia. Sin embargo, puedo darte un resumen sobre **Marte** basado en mi conocimiento general:

Marte es el cuarto planeta del Sistema Solar y el segundo más pequeño después de Mercurio. Es conocido como el "Planeta Rojo" debido al óxido de hierro presente en su superficie, que le da ese característico color rojizo.

> Finished chain.
Respuesta 1: Parece que hay un problema técnico con la herramienta para obtener información de Wikipedia. Sin embargo, puedo darte un resumen sobre **Marte** basado en mi conocimiento general:

Marte es el cuarto planeta del Sistema Solar y el segundo más pequeño después de Mercurio. Es conocido como el "Planeta Rojo" debido al óxido de hierro presente en su superficie, que le da ese característico color rojizo.
Segunda pregunta de seguimiento...


> Entering new AgentExecutor chain...



Invoking: `get_wikipedia_summary` with `{'query': 'Lunas de Marte'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)

Marte tiene **dos lunas**, llamadas **Fobos** y **Deimos**. Ambas son pequeñas y de forma irregular, y se cree que podrían ser asteroides capturados por la gravedad del planeta. Fobos es la más grande y cercana a Marte, mientras que Deimos es más pequeña y está más lejos.

> Finished chain.
Respuesta 2: Marte tiene **dos lunas**, llamadas **Fobos** y **Deimos**. Ambas son pequeñas y de forma irregular, y se cree que podrían ser asteroides capturados por la gravedad del planeta. Fobos es la más grande y cercana a Marte, mientras que Deimos es más pequeña y está más lejos.


Como puedes ver, la función `chat_with_agent_buffer` ahora se encarga de cargar y guardar el historial, haciendo el proceso mucho más limpio.

### 4. Gestionando el Contexto con `ConversationBufferWindowMemory`

Para evitar que el historial crezca sin control, podemos usar `ConversationBufferWindowMemory`. Esta memoria solo conserva un número `k` de las últimas interacciones.

Esto es un buen equilibrio entre tener contexto y no sobrecargar al LLM.

In [6]:
# LangChain v1: persistencia de LangGraph (RunnableWithMessageHistory, el paso
# intermedio que usa este notebook, también quedó deprecado)
from langchain_classic.memory import ConversationBufferWindowMemory

# Creamos una nueva memoria, esta vez con k=1 (solo recuerda el último par de mensajes)
memory_window = ConversationBufferWindowMemory(k=1, memory_key="chat_history", return_messages=True)

def chat_with_agent_window(query: str):
    history = memory_window.load_memory_variables({})["chat_history"]
    response = agent_executor.invoke({"input": query, "chat_history": history})
    memory_window.save_context({"input": query}, {"output": response["output"]})
    return response["output"]

/var/folders/hc/s32qf6c501b8c_vjpqkvxdpw0000gn/T/ipykernel_8889/3895951803.py:6: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory_window = ConversationBufferWindowMemory(k=1, memory_key="chat_history", return_messages=True)


In [7]:
print("Pregunta 1: ¿Quién fue Albert Einstein?")
chat_with_agent_window("¿Quién fue Albert Einstein?")

print("Pregunta 2: ¿Y Niels Bohr?")
chat_with_agent_window("¿Y Niels Bohr?")

print("Pregunta 3: ¿De qué trataba la primera pregunta que te hice?")
response3 = chat_with_agent_window("¿De qué trataba la primera pregunta que te hice?")
print(f"Respuesta 3: {response3}")

Pregunta 1: ¿Quién fue Albert Einstein?


> Entering new AgentExecutor chain...



Invoking: `get_wikipedia_summary` with `{'query': 'Albert Einstein'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)


Invoking: `get_wikipedia_summary` with `{'query': 'Albert Einstein'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)

...

Ocurrió un error: Expecting value: line 1 column 1 (char 0)


Invoking: `get_wikipedia_summary` with `{'query': 'Niels Bohr'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)

Niels Bohr fue un físico danés pionero en la teoría cuántica y el modelo atómico que lleva su nombre, el cual explicó la estructura del átomo y sentó las bases de la mecánica cuántica. También contribuyó al desarrollo de la física nuclear y fue un defensor del diálogo científico internacional.

> Finished chain.
Pregunta 3: ¿De qué trataba la primera pregunta que te hice?


> Entering new AgentExecutor chain...


La primera pregunta que me hiciste fue: **"¿Y Niels Bohr?"**, lo que implica que querías que te diera información sobre este científico.

> Finished chain.
Respuesta 3: La primera pregunta que me hiciste fue: **"¿Y Niels Bohr?"**, lo que implica que querías que te diera información sobre este científico.


El agente probablemente no podrá responder a la tercera pregunta, porque con `k=1`, la interacción sobre Einstein ya fue descartada de la memoria para dar paso a la de Niels Bohr.

### 5. Ahorrando Tokens con `ConversationSummaryMemory`

Esta es la estrategia más avanzada. En lugar de guardar los mensajes completos, utiliza un LLM para crear un resumen de la conversación a medida que avanza. En cada nueva interacción, el resumen se actualiza y se pasa como contexto.

Es ideal para conversaciones muy largas donde los detalles específicos se vuelven menos importantes que el contexto general.

In [8]:
# LangChain v1: persistencia de LangGraph (RunnableWithMessageHistory, el paso
# intermedio que usa este notebook, también quedó deprecado)
from langchain_classic.memory import ConversationSummaryMemory

# Esta memoria necesita un LLM para hacer los resúmenes
memory_summary = ConversationSummaryMemory(llm=llm, memory_key="chat_history", return_messages=True)

def chat_with_agent_summary(query: str):
    history = memory_summary.load_memory_variables({})["chat_history"]
    response = agent_executor.invoke({"input": query, "chat_history": history})
    memory_summary.save_context({"input": query}, {"output": response["output"]})
    return response["output"]

/var/folders/hc/s32qf6c501b8c_vjpqkvxdpw0000gn/T/ipykernel_8889/193633793.py:6: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory_summary = ConversationSummaryMemory(llm=llm, memory_key="chat_history", return_messages=True)


In [9]:
print("Pregunta 1: Necesito organizar un viaje a Japón. ¿Cuál es la mejor época para ir?")
chat_with_agent_summary("Necesito organizar un viaje a Japón. ¿Cuál es la mejor época para ir?")

print("Pregunta 2: Suena bien. ¿Qué ciudades me recomiendas visitar en un primer viaje?")
chat_with_agent_summary("¿Qué ciudades me recomiendas visitar en un primer viaje?")

print("Pregunta 3: ¿Sobre qué estábamos hablando?")
response3_summary = chat_with_agent_summary("¿Sobre qué estábamos hablando?")
print(f"Respuesta 3: {response3_summary}")

Pregunta 1: Necesito organizar un viaje a Japón. ¿Cuál es la mejor época para ir?


> Entering new AgentExecutor chain...



Invoking: `get_wikipedia_summary` with `{'query': 'mejor época para viajar a Japón'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)


Invoking: `get_wikipedia_summary` with `{'query': 'mejor época para visitar Japón'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)

...

Pregunta 2: Suena bien. ¿Qué ciudades me recomiendas visitar en un primer viaje?


> Entering new AgentExecutor chain...


Para un primer viaje a Japón, te recomendaría visitar estas ciudades, cada una con su propio encanto y experiencias únicas:

1. **Tokio**: La capital es una mezcla vibrante de tradición y modernidad. No te pierdas lugares como el Templo Senso-ji en Asakusa, el barrio de Shibuya con su famoso cruce, y el moderno distrito de Shibuya o Akihabara para los amantes de la tecnología y el anime. Además, es ideal para probar la gastronomía japonesa en sus múltiples restaurantes.

2. **Kioto**: Conocida como la capital cultural de Japón, Kioto es imprescindible para ver templos históricos, santuarios como el Fushimi Inari con sus miles de torii rojos, y jardines tradicionales. Si visitas en primavera u otoño, podrás disfrutar de los cerezos en flor o el follaje otoñal, respectivamente.

3. **Osaka**: Perfecta para los amantes de la comida, Osaka es famosa por su street food y restaurantes. No dejes de visitar el castillo de Osaka, el acuario Kaiyukan y el barrio de Dotonbori, lleno de luces y vi

Pregunta 3: ¿Sobre qué estábamos hablando?


> Entering new AgentExecutor chain...


Estábamos hablando sobre **Japón**. Primero, te recomendé la mejor época para visitarlo según tus preferencias (primavera, otoño, verano o invierno), destacando el otoño como una opción ideal para evitar multitudes y disfrutar del follaje otoñal. Luego, te sugerí **cinco ciudades imprescindibles para un primer viaje a Japón**: Tokio, Kioto, Osaka, Hiroshima y Nara, además de mencionar opciones como Hakone y Kanazawa. También te ofrecí ayuda para planificar un itinerario.

Si quieres, puedo profundizar en algún destino, darte más detalles sobre actividades o ayudarte a organizar un plan de viaje. ¿En qué más te gustaría saber? 😊

> Finished chain.


Respuesta 3: Estábamos hablando sobre **Japón**. Primero, te recomendé la mejor época para visitarlo según tus preferencias (primavera, otoño, verano o invierno), destacando el otoño como una opción ideal para evitar multitudes y disfrutar del follaje otoñal. Luego, te sugerí **cinco ciudades imprescindibles para un primer viaje a Japón**: Tokio, Kioto, Osaka, Hiroshima y Nara, además de mencionar opciones como Hakone y Kanazawa. También te ofrecí ayuda para planificar un itinerario.

Si quieres, puedo profundizar en algún destino, darte más detalles sobre actividades o ayudarte a organizar un plan de viaje. ¿En qué más te gustaría saber? 😊


El agente debería poder responder correctamente, ya que la memoria no contiene los mensajes literales, sino un resumen de que el usuario está planeando un viaje a Japón.

## Conclusiones y Transición al Módulo IL2.3

Hemos visto cómo las clases de memoria de LangChain nos permiten crear agentes conversacionales mucho más robustos y eficientes, superando los desafíos de la gestión manual y el tamaño del contexto.

**Resumen de Estrategias de Memoria:**
- **`ConversationBufferMemory`**: Simple y completa. Ideal para conversaciones cortas donde cada detalle importa.
- **`ConversationBufferWindowMemory`**: Eficiente y balanceada. Perfecta para chatbots de servicio al cliente que necesitan contexto reciente.
- **`ConversationSummaryMemory`**: Ahorradora de tokens y escalable. La mejor opción para asistentes de largo plazo y análisis de conversaciones extensas.

Con un agente que puede recordar de manera efectiva, hemos completado una pieza fundamental del rompecabezas. Sin embargo, las conversaciones del mundo real a menudo implican tareas que no se resuelven con una sola herramienta o en un solo paso.

**¿Qué pasa si el agente necesita buscar información, luego calcular algo con esa información y finalmente enviar un correo electrónico?**

Esto requiere que el agente pueda **planificar** una secuencia de acciones. Debe ser capaz de descomponer un objetivo complejo en pasos más pequeños y ejecutarlos en el orden correcto.

Esta es la transición perfecta al **Módulo IL2.3: Planificación y Orquestación de Agentes**, donde exploraremos cómo los agentes pueden crear, seguir y adaptar planes para resolver problemas complejos y multi-paso.